# Inference speed: HF Transformers vs vLLM

Benchmark single-prompt latency and throughput on the same
Llama-3.2-1B-Instruct checkpoint across:

1. HF Transformers (eager generation)
2. vLLM with `gpu_memory_utilization=0.2`
3. vLLM with `gpu_memory_utilization=0.7`

Decoding is stochastic (`do_sample=True` / `temperature > 0`) on
both backends, with matching `temperature` and `top_p`. Each timed
iteration uses a fresh seed `base_seed + i` — outputs differ from
run to run but are reproducible across re-executions of the cell.
Token counts vary per iteration since the sampled completions hit
EOS at different points; throughput (tokens / second) is the
robust metric to compare.

Each backend includes one untimed warmup pass to exclude cudagraph
capture / JIT compilation from the latency.

Note: `gpu_memory_utilization` mainly affects vLLM's KV-cache pool
size, which matters for *concurrent* requests. On a single prompt
the two vLLM settings should land within noise of each other.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import warnings
warnings.filterwarnings("ignore")

import gc
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

In [2]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

dataset_dir = base_dir + "/prm800k/math_splits"

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"

In [3]:
# Benchmark prompt + decoding config
prompt = (
    r'If $f(x) = \frac{3x-2}{x-2}$, what is the value of '
    r'$f(-2) + f(-1) + f(0)$? Express your answer as a common fraction.'
)
max_new_tokens = 1024
num_runs = 10

# Stochastic decoding — same params on both backends for a fair comparison
temperature = 0.8
top_p = 0.95
base_seed = 123    # iteration i uses seed = base_seed + i

In [ ]:
import sys
sys.path.append("..")

from unittests.utils import gpu_mem_used_gb, measure_inference

## HF Transformers (baseline)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(llm_dir)
model_hf = AutoModelForCausalLM.from_pretrained(
    llm_dir,
    dtype="float16",
    device_map="cuda:0",
)
model_hf.eval()

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')
print(model_hf.dtype)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

#--- GPU memory used: 6.12 GB
torch.float16


In [ ]:
latency_hf, throughput_hf, avg_tokens_hf, text_hf = measure_inference(
    "hf", model_hf, tokenizer, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"HF Transformers   - latency: {latency_hf:.4f}s, "
    f"throughput: {throughput_hf:.2f} tok/s, "
    f"avg tokens: {avg_tokens_hf:.1f}"
)

In [ ]:
# Free HF before loading vLLM so they don't fight over the GPU
del model_hf
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

#--- GPU memory used: 0.30 GB


## vLLM with `gpu_memory_utilization=0.2`

Small KV-cache pool — enough headroom for one prompt but not for
concurrent batching at long contexts.

In [5]:
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.3,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.15s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.64s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.71s/it]

Capturing CUDA graphs (mixed pref

#--- GPU memory used: 10.75 GB


In [6]:
latency_v02, throughput_v02, avg_tokens_v02, text_v02 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.2  - latency: {latency_v02:.4f}s, "
    f"throughput: {throughput_v02:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v02:.1f}"
)

vLLM gpu_mem=0.2  - latency: 4.2189s, throughput: 114.30 tok/s, avg tokens: 482.2


In [ ]:
# Free the first vLLM engine before reloading at a higher pool size
del llm_vllm
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

[rank0]:[W612 10:52:59.355156664 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


#--- GPU memory used: 0.30 GB


## vLLM with `gpu_memory_utilization=0.7`

Large KV-cache pool. For a single prompt this should match the
0.2 setting within noise; the difference shows up under concurrent
batching (more sequences live in cache at once).

In [ ]:
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.7,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.17s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.64s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.72s/it]

Capturing CUDA graphs (mixed pref

#--- GPU memory used: 23.47 GB


In [ ]:
latency_v07, throughput_v07, avg_tokens_v07, text_v07 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.7  - latency: {latency_v07:.4f}s, "
    f"throughput: {throughput_v07:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v07:.1f}"
)

vLLM gpu_mem=0.7  - latency: 4.2620s, throughput: 113.14 tok/s, avg tokens: 482.2


## Summary

In [ ]:
header = f"{'Backend':<22}{'Latency (s)':>14}{'Tok/s':>12}{'Avg tok':>12}"
print(header)
print('-' * len(header))
rows = [
    ('HF Transformers',  latency_hf,  throughput_hf,  avg_tokens_hf),
    ('vLLM gpu_mem=0.2', latency_v02, throughput_v02, avg_tokens_v02),
    ('vLLM gpu_mem=0.7', latency_v07, throughput_v07, avg_tokens_v07),
]
for name, lat, tput, ntok in rows:
    print(f"{name:<22}{lat:>14.4f}{tput:>12.2f}{ntok:>12.1f}")

# print(f"\nSample completion (HF, last run):\n{text_hf[:400]}...")

Backend                  Latency (s)       Tok/s     Avg tok
------------------------------------------------------------
HF Transformers              17.8811       26.41       472.3
vLLM gpu_mem=0.2              4.2524      113.39       482.2
vLLM gpu_mem=0.7              4.2620      113.14       482.2
